# Wildfire data exploration — MTBS + LANDFIRE

A starter notebook for the `fire-prediction-model-csci4502` project. Run cells top-to-bottom. It uses **MTBS fire-perimeter vectors** and checks the **LANDFIRE raster/archive** files without assuming exactly which variables were downloaded.

**In VS Code:** Open this `.ipynb`, click **Select Kernel** (top right), select your project Python environment, then click **Run All** or execute cells individually. Execute from the project folder; the notebook automatically finds the `data/raw` directory.

## 1. Imports and find your data

In [1]:
# In a VS Code terminal, run once if needed:
# python -m pip install -r requirements.txt
# python -m pip install ipykernel geopandas matplotlib rasterio

from pathlib import Path
import zipfile
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt

# Works if the notebook lives in the project root or notebooks/.
possible_roots = [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]
PROJECT_ROOT = next((p for p in possible_roots if (p / "data" / "raw").exists()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Could not find data/raw. Open the notebook from your project folder.")
RAW = PROJECT_ROOT / "data" / "raw"
print("Project:", PROJECT_ROOT.resolve())
print("Raw data:", RAW.resolve())

Project: C:\Users\seans\fire-prediction-model-csci4502
Raw data: C:\Users\seans\fire-prediction-model-csci4502\data\raw


## 2. Inventory: what was downloaded?

In [2]:
rows = []
for path in sorted(RAW.rglob("*")):
    if path.is_file():
        rows.append({"file": str(path.relative_to(RAW)),
                     "type": path.suffix.lower(),
                     "size_MB": round(path.stat().st_size / 1_000_000, 2)})
files = pd.DataFrame(rows)
display(files)

,file,type,size_MB
0,landfire_east\j70a9e7ff46b44782b055a330f88bd8c...,.tfw,0.00
1,landfire_east\j70a9e7ff46b44782b055a330f88bd8c...,.tif,0.74
2,landfire_east\j70a9e7ff46b44782b055a330f88bd8c...,.xml,0.01
3,landfire_east.zip,.zip,0.75
4,mtbs\mtbs_perimeter_data.zip,.zip,389.93
5,mtbs\mtbs_perims_DD.cpg,.cpg,0.00
6,mtbs\mtbs_perims_DD.dbf,.dbf,43.34
7,mtbs\mtbs_perims_DD.prj,.prj,0.00
8,mtbs\mtbs_perims_DD.shp,.shp,616.60
9,mtbs\mtbs_perims_DD.shx,.shx,0.25


## 3. Inspect MTBS wildfire perimeters

In [ ]:
shp = RAW / "mtbs" / "mtbs_perims_DD.shp"
gpkg = RAW / "mtbs_2020_2020.gpkg"

if shp.exists():
    fires = gpd.read_file(shp)
    source = shp
elif gpkg.exists():
    fires = gpd.read_file(gpkg)
    source = gpkg
else:
    raise FileNotFoundError("No MTBS shapefile or GeoPackage found under data/raw")

print("Source:", source.relative_to(PROJECT_ROOT))
print("Number of fire records:", len(fires))
print("Coordinate reference system (CRS):", fires.crs)
print("Geometry types:", fires.geometry.geom_type.value_counts().to_dict())
print("Columns:", fires.columns.tolist())
display(fires.head())

## 4. Missing data and summary statistics

In [ ]:
display(pd.DataFrame({"dtype": fires.dtypes.astype(str),
                      "missing": fires.isna().sum(),
                      "missing_pct": (fires.isna().mean() * 100).round(1)}))
display(fires.describe(include="all").transpose().head(40))

## 5. Map the fires (static)

In [ ]:
# Reproject for easier geographic display; geometry plotting works even without a basemap.
plot_fires = fires.to_crs("EPSG:4326") if fires.crs else fires
fig, ax = plt.subplots(figsize=(12, 8))
plot_fires.plot(ax=ax, color="orangered", edgecolor="darkred", linewidth=0.25, alpha=0.5)
ax.set_title("MTBS wildfire perimeters")
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
plt.tight_layout()
plt.show()

## 6. Which fires are largest? (calculated area)

To estimate area, transform to an equal-area projection first. This gives **approximate polygon area**, not burn severity or exact acres reported by MTBS.

In [ ]:
if fires.crs is None:
    print("This layer does not declare a CRS; skip area calculations until that is resolved.")
else:
    area_fires = fires.to_crs("EPSG:5070").copy()  # US National Atlas Equal Area
    area_fires["area_acres_est"] = area_fires.geometry.area / 4046.8564224
    display(area_fires.drop(columns="geometry").sort_values("area_acres_est", ascending=False).head(10))
    fig, ax = plt.subplots(figsize=(9, 4))
    area_fires["area_acres_est"].dropna().plot.hist(bins=40, ax=ax)
    ax.set_xlabel("Fire polygon area (approx. acres)")
    ax.set_title("Fire perimeter area distribution")
    plt.tight_layout()
    plt.show()

## 7. Inspect the other MTBS GeoPackage

A GeoPackage can hold multiple layers. Look up the layer names before loading one.

In [ ]:
if gpkg.exists():
    print("GeoPackage layers:")
    try:
        import fiona
        layers = fiona.listlayers(gpkg)
    except (ImportError, AttributeError):
        import pyogrio
        layers = pyogrio.list_layers(gpkg)[:, 0].tolist()
    print(layers)
    for layer in layers:
        sample = gpd.read_file(gpkg, layer=layer, rows=5)
        print(f"\nLayer: {layer} | CRS: {sample.crs} | columns: {sample.columns.tolist()}")
        display(sample.head())
else:
    print("No GeoPackage found")

## 8. Inspect LANDFIRE files and archive contents

LANDFIRE commonly supplies raster datasets (such as `.tif` / `.img`). ZIP files may contain metadata and rasters. This cell identifies what is actually present.

In [ ]:
landfire_dir = RAW / "landfire_east"
landfire_files = list(landfire_dir.rglob("*")) if landfire_dir.exists() else []
print("Extracted LANDFIRE files:")
for f in landfire_files:
    if f.is_file():
        print(" ", f.relative_to(RAW), f"({f.stat().st_size / 1e6:.1f} MB)")

archives = [p for p in [RAW / "landfire_east.zip", *landfire_files] if p.is_file() and p.suffix.lower() == ".zip"]
for archive in archives:
    print(f"\nZIP: {archive.relative_to(RAW)}")
    if zipfile.is_zipfile(archive):
        with zipfile.ZipFile(archive) as z:
            names = z.namelist()
            print("Entries:", len(names))
            print("First 25 entries:", *names[:25], sep="\n  ")
    else:
        print("Not a valid ZIP archive")

## 9. Preview a LANDFIRE raster if one exists

If no raster is detected, inspect/extract the ZIP contents first. Large rasters are downsampled for display to avoid running out of memory.

In [ ]:
from rasterio.enums import Resampling
import rasterio
import numpy as np

raster_extensions = {".tif", ".tiff", ".img", ".vrt"}
rasters = [p for p in landfire_files if p.is_file() and p.suffix.lower() in raster_extensions]
if not rasters:
    print("No extracted raster found. Check the ZIP list above and extract any raster-containing archive.")
else:
    raster_path = rasters[0]
    print("Raster:", raster_path.relative_to(RAW))
    with rasterio.open(raster_path) as src:
        print("CRS:", src.crs, "size:", src.width, "x", src.height,
              "bands:", src.count, "nodata:", src.nodata)
        max_dim = 1000
        factor = min(1.0, max_dim / max(src.width, src.height))
        height = max(1, int(src.height * factor))
        width = max(1, int(src.width * factor))
        data = src.read(1, out_shape=(height, width), resampling=Resampling.nearest, masked=True)
        fig, ax = plt.subplots(figsize=(11, 7))
        im = ax.imshow(data, cmap="viridis", interpolation="nearest")
        fig.colorbar(im, ax=ax, label="Raster value / class")
        ax.set_title(f"LANDFIRE preview: {raster_path.name}")
        ax.set_axis_off()
        plt.tight_layout()
        plt.show()
        print("Raster value codes may be categories—consult LANDFIRE metadata for interpretation.")

## 10. Next analysis ideas

- Compare wildfire perimeter **counts and estimated areas by year** if the MTBS attributes include a year/date field.
- Create a **Colorado subset** by spatially intersecting with a state boundary (not just a bounding box).
- Overlay **LANDFIRE vegetation/fuel rasters** and fire perimeters once raster CRS and class meanings are confirmed.
- Avoid using future fire perimeter data as a predictor of earlier fires (data leakage) in your prediction model.

**Tip:** To explore any GeoDataFrame, use `display(fires.head())`, `fires.columns`, or `fires.plot()` in a new cell.